# 16.9 Attention 中間矩陣為何佔空間？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：邊切邊裝盤：不存整張注意力中間表**

數學結果等價；Python示例不是Flash GPU kernel。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/online_softmax.svg")))

**先想一想：**沒有存整張T×T表，還能求完整softmax平均嗎？

FlashAttention像一邊切菜一邊裝盤，不把整張巨大中間表來回搬存儲。關鍵是分塊與online softmax仍得到同數學結果，不是近似丟掉注意力。

**把直覺寫成數學：**維護每行running最大值、分母與加權分子，換最大值時同時重新縮放。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.efficiency import online_attention

q, k, v = torch.randn(3, 4), torch.randn(7, 4), torch.randn(7, 2)
reference = (q @ k.T / (4**0.5)).softmax(-1) @ v
streamed = online_attention(q, k, v, block_size=2)
print((reference - streamed).abs().max().item())
assert torch.allclose(reference, streamed, atol=1e-6)

**如何讀結果：**這份Python CPU數值版不是Flash GPU kernel，反向也不具其省memory實現。實際用相容GPU的SDPA後端測峯值。

**只改一件事：**只把block_size改爲3，結果仍應等價。
